In [0]:
%run "/Workspace/Optum-DBx/Project-Optum-DBx/Dbx-Transformations/Prod/Connectors_Prod"

In [0]:
%run "/Workspace/Optum-DBx/Project-Optum-DBx/Dbx-Transformations/Prod/Generic_Prod"

In [0]:
#Importing neccessary libraries
from pyspark.sql.functions import *

In [0]:
#Calling the function connect to ADLS storage from Connectors
adls_connect()

In [0]:
#Listing all the files in Bronze layer
list_bronze_files()

In [0]:
#Reading Subgroup.csv file from Bronze layer
subscriber_df = read_bronze_file_csv("subscriber")

In [0]:
check_missing_values(subscriber_df,subscriber_df.columns)

In [0]:
check_string_value_as_nan(subscriber_df)

Transformation Layer

In [0]:
# Filling Missing Values
subscriber_df = subscriber_df.fillna({"first_name":"Visitor/NA","Elig_ind":"N"})

In [0]:
subscriber_df = subscriber_df.drop("Phone")

In [0]:
# Deriving Age from Birth Date
subscriber_df = subscriber_df.withColumn("subscriber_age",floor(months_between(current_date(),subscriber_df.Birth_date)/12))

In [0]:
subscriber_df = subscriber_df.drop("Birth_date")

In [0]:
# Finding Null values (For Reverse Engineering)
display(subscriber_df.select("*").filter(col("Subgrp_id").isNull()))

In [0]:
# Filling sub group id
subscriber_df = subscriber_df.withColumn("Subgrp_id", when((col("Subgrp_id").isNull()) & (col("sub_id")=="SUBID10022"), "S110").when((col("Subgrp_id").isNull()) & (col("sub_id")=="SUBID10049"), "S107").otherwise(col("Subgrp_id")))

Writing to Silver layer

In [0]:
#Writing transformed dataframe into Silver layer
write_to_silver(subscriber_df,"subscriber_S.csv")